Import dan membuat SparkSession

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.functions import sum as spark_sum, count, avg

spark = SparkSession.builder \
    .appName("TugasMandiriPertemuan4") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/16 16:20:37 WARN Utils: Your hostname, zaydann resolves to a loopback address: 127.0.1.1; using 10.90.70.192 instead (on interface wlo1)
26/09/16 16:20:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 16:20:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


A. Membaca dan Eksplorasi Awal

In [2]:
df = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",
    header=True,
    inferSchema=True
)

print("Data berhasil dibaca dari HDFS.")

Data berhasil dibaca dari HDFS.


In [3]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)



In [4]:
print("Jumlah baris:", df.count())

Jumlah baris: 1000


In [5]:
df.show(10)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|
|ORD-3004|2026-09-10 00:00:00|        Rumah Tangga|Yogyakarta|          10|       60000|         E-Wallet|   4.0|
|ORD-3005|2026-09-09 00:00:00|             Fashion| Purworejo|           5|       20000|

B. Menangani Data Kosong

In [6]:
jumlah_rating_kosong = df.filter(
    col("rating").isNull()
).count()

print("Jumlah rating kosong:", jumlah_rating_kosong)

Jumlah rating kosong: 204


In [8]:
rata_rata_rating = df.select(
    avg("rating").alias("rata_rata_rating")
).collect()[0]["rata_rata_rating"]

print("Rata-rata rating:", rata_rata_rating)

Rata-rata rating: 4.1457286432160805


In [9]:
df = df.na.fill({
    "rating": rata_rata_rating
})

print("Data kosong pada rating sudah ditangani.")

Data kosong pada rating sudah ditangani.


In [10]:
jumlah_rating_kosong = df.filter(
    col("rating").isNull()
).count()

print("Jumlah rating kosong setelah ditangani:", jumlah_rating_kosong)

Jumlah rating kosong setelah ditangani: 0


Pada kolom rating terdapat beberapa data kosong yang perlu diisi agar analisis tetap berjalan dengan baik. Data kosong tersebut diisi menggunakan nilai rata-rata rating dengan fungsi `df.na.fill()`. Dengan cara ini, semua data tetap digunakan tanpa mengurangi jumlah transaksi yang ada.


C. Transformasi Data

In [11]:
df = df.withColumn(
    "total_pendapatan",
    col("unit_terjual") * col("harga_satuan")
)

In [12]:
from pyspark.sql.functions import when

df = df.withColumn(
    "tier_transaksi",
    when(
        col("total_pendapatan") > 500000,
        "Besar"
    ).otherwise("Kecil")
)

In [13]:
df.select(
    "order_id",
    "unit_terjual",
    "harga_satuan",
    "total_pendapatan",
    "tier_transaksi"
).show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Kecil|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Kecil|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Kecil|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Kecil|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



D. Analisis dengan GroupBy

In [14]:
pendapatan_kategori = df.groupBy(
    "kategori"
).agg(
    spark_sum("total_pendapatan").alias("total_pendapatan")
).orderBy(
    col("total_pendapatan").desc()
)

pendapatan_kategori.show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|        Rumah Tangga|       138665000|
|   Makanan & Minuman|       131890000|
|Kesehatan & Kecan...|       128595000|
|            Olahraga|       126650000|
|             Fashion|       124075000|
|          Elektronik|       110295000|
+--------------------+----------------+



In [15]:
kategori_tertinggi = pendapatan_kategori.first()

print("Kategori dengan total pendapatan tertinggi:")
print(kategori_tertinggi["kategori"])
print(
    "Total pendapatan:",
    kategori_tertinggi["total_pendapatan"]
)

Kategori dengan total pendapatan tertinggi:
Rumah Tangga
Total pendapatan: 138665000


In [16]:
transaksi_besar = df.filter(
    col("tier_transaksi") == "Besar"
)

In [17]:
jumlah_besar_per_kota = transaksi_besar.groupBy(
    "kota"
).count().orderBy(
    col("count").desc()
)

jumlah_besar_per_kota.show()

+----------+-----+
|      kota|count|
+----------+-----+
|      Solo|   92|
|  Magelang|   78|
|   Kebumen|   78|
|Yogyakarta|   75|
| Purworejo|   66|
|  Semarang|   65|
+----------+-----+



In [18]:
kota_terbanyak = jumlah_besar_per_kota.first()

print("Kota dengan transaksi tier Besar terbanyak:")
print(kota_terbanyak["kota"])
print(
    "Jumlah transaksi:",
    kota_terbanyak["count"]
)

Kota dengan transaksi tier Besar terbanyak:
Solo
Jumlah transaksi: 92


In [19]:
rata_rating_metode = df.groupBy(
    "metode_pembayaran"
).agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(
    col("rata_rata_rating").desc()
)

rata_rating_metode.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



E. Menyimpan Hasil ke HDFS

In [20]:
df.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

In [21]:
output_path = "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan"

df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(output_path)

print("Data hasil olahan berhasil disimpan ke HDFS.")

Data hasil olahan berhasil disimpan ke HDFS.


In [22]:
!hdfs dfs -ls /user/mahasiswa/tugas4/hasil_olahan

Found 2 items
-rw-r--r--   3 zaydann supergroup          0 2026-09-16 16:23 /user/mahasiswa/tugas4/hasil_olahan/_SUCCESS
-rw-r--r--   3 zaydann supergroup     100356 2026-09-16 16:23 /user/mahasiswa/tugas4/hasil_olahan/part-00000-f8d44788-2da0-459d-9660-5a6ec951d554-c000.csv


In [23]:
df_hasil = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas4/hasil_olahan",
    header=True,
    inferSchema=True
)

df_hasil.show(5)

+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|total_pendapatan|tier_transaksi|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+----------------+--------------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|          270000|         Kecil|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|          600000|         Besar|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|           8|       60000|         E-Wallet|   3.0|          480000|         Kecil|
|ORD-3003|2026-09-09 00:00:00|   Makanan & Minuman|  Semarang|           6|      350000|    Transfer Bank|   4.0|         21

Spark membagi data menjadi beberapa partisi agar prosesnya bisa dilakukan secara paralel. Karena itu, hasil penyimpanan dapat muncul dalam beberapa part file. Hal ini normal dan merupakan bagian dari cara Spark memproses data.


In [24]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
